<a href="https://colab.research.google.com/github/Aleeza-Nadeem/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aleeza-Nadeem/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

**What I built:** one row per page. Inputs come only from the *previous* 30 days plus static page fields; the thing to predict (clicks in the *next* 30 days) is kept separate and never enters the feature table. Rows missing key fields, or with zero impressions in the previous 30 days, are dropped. Missing values in the other numeric fields are counted below and filled with medians at modeling time, computed on training rows only. `content_type` labels are renamed `type_1`, `type_2` and so on so no brand-like names appear in outputs.

In [1]:
import pandas as pd, numpy as np

url = "https://huggingface.co/datasets/FlyRank/internship-starter/resolve/main/content_refresh_anonymized.csv"
df = pd.read_csv(url)

need = ["clicks_last_30d", "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d", "avg_position", "client_id"]
d = df.dropna(subset=need).copy()
d = d[d.impressions_prev_30d > 0].reset_index(drop=True)
print("rows used:", len(d), "of", len(df), "| clients:", d.client_id.nunique())

num = ["impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d", "search_volume", "competition", "cpc",
       "word_count", "content_age_days", "days_since_last_update", "avg_position"]
cat = ["content_type", "main_intent"]

# anonymize content_type so no brand-like names appear in outputs
ct = d["content_type"].fillna("unknown")
d["content_type"] = ct.map({v: f"type_{i+1}" for i, v in enumerate(ct.value_counts().index)})

X = pd.concat([d[num], pd.get_dummies(d[cat].fillna("unknown"), dtype=float)], axis=1)
y = (d.clicks_last_30d >= d.clicks_last_30d.quantile(0.80)).astype(int)   # label kept OUT of X
print("feature table:", X.shape, "| label positive rate:", round(y.mean(), 3))
X.head(3)


rows used: 26612 of 30000 | clients: 31
feature table: (26612, 18) | label positive rate: 0.212


,impressions_prev_30d,clicks_prev_30d,sessions_prev_30d,search_volume,competition,cpc,word_count,content_age_days,days_since_last_update,avg_position,content_type_type_1,content_type_type_2,content_type_type_3,main_intent_commercial,main_intent_informational,main_intent_navigational,main_intent_transactional,main_intent_unknown
0,987,13,9,10.0,0.67,2.05,3221.0,187,20,10.6,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,5915,1,2,90.0,0.01,0.05,2481.0,445,25,20.3,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,6089,3,3,0.0,0.00,0.00,3515.0,141,20,36.5,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Feature | Meaning | Missing values | Exists before I predict? |
|---|---|---|---|
| `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d` | search impressions, clicks and visits in the previous 30 days | rows missing them are dropped | yes, earlier window |
| `avg_position` | average search position over 90 days | rows missing it are dropped | mostly: the 90 days overlap the label window (caveat) |
| `search_volume`, `competition`, `cpc` | keyword demand, difficulty and price estimates for the page | median fill (training rows only) | yes, static fields |
| `word_count` | length of the page | median fill | yes |
| `content_age_days`, `days_since_last_update` | page age and time since last edit | median fill | mostly: measured at the snapshot, so an edit inside the label window could shift them (caveat) |
| `content_type`, `main_intent` | page type (anonymized) and main search intent | "unknown" category | yes |

Categorical columns become one 0/1 column per category. No other features were engineered; log-scaling and standardizing happen inside the model pipeline.

In [2]:
miss = pd.DataFrame({
    "missing_pct": (d[num + cat].isna().mean() * 100).round(2),
    "available_when": ["previous 30 days"] * 3 + ["static page field"] * 3 + ["static page field"] * 3 + ["90-day window (overlaps label)"] + ["static page field"] * 2})
miss.loc["avg_position", "available_when"] = "90-day window (overlaps label)"
miss.loc[["content_age_days", "days_since_last_update"], "available_when"] = "snapshot date (check timing)"
miss


,missing_pct,available_when
impressions_prev_30d,0.00,previous 30 days
clicks_prev_30d,0.00,previous 30 days
sessions_prev_30d,0.00,previous 30 days
search_volume,4.98,static page field
competition,4.98,static page field
cpc,4.98,static page field
word_count,27.98,static page field
content_age_days,0.00,snapshot date (check timing)
days_since_last_update,0.00,snapshot date (check timing)
avg_position,0.00,90-day window (overlaps label)


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

**How I attacked my own features:**
1. **Name scan:** list every column in the release that comes from the label window, is rule-made (flags, health score, trend) or identifies something, and confirm none is in the feature table.
2. **Single-feature test:** score each feature alone against the label; an AUC above 0.95 is a flag I must explain, not ignore.
3. **Shuffled-label test:** train on a shuffled label with clients held out; AUC should be near 0.5.
4. **Duplicate pages:** no page may appear in both train and test.
5. **Privacy scan:** look for columns that could hold names, addresses or search queries.

Results are written under the code cell and summarized in section 4.

In [3]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# 1) name scan over EVERY column in the release (names only, no values)
label_window = [c for c in df.columns if "last_30d" in c]
rule_made = [c for c in df.columns if any(w in c for w in ["needs_", "is_", "ai_opportunity", "health_score", "trend", "_tier", "direction", "flag"])]
ids = [c for c in ["client_id", "content_id"] if c in df.columns]
privacy_words = ["url", "query", "keyword", "domain", "name", "title", "email"]
privacy_cols = [c for c in df.columns if any(w in c.lower() for w in privacy_words)]
print("label-window columns:", label_window)
print("rule-made / label-derived columns:", rule_made)
print("identifier columns:", ids)
print("possible privacy columns (names only):", privacy_cols if privacy_cols else "none")
overlap = [c for c in X.columns if c in label_window + rule_made + ids]
print("excluded columns that leaked into X:", overlap if overlap else "none")

# 2) grouped split (same as the model notebook), then single-feature AUC on held-out clients
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr, te = next(gss.split(X, y, d.client_id))
cut = d.loc[tr, "clicks_last_30d"].quantile(0.80)
y = (d.clicks_last_30d >= cut).astype(int)
single = {}
for col in X.columns:
    v = X.iloc[te][col].values
    if np.nanstd(v) > 0:
        a = roc_auc_score(y.iloc[te], np.nan_to_num(v)); single[col] = max(a, 1 - a)
single = pd.Series(single).sort_values(ascending=False)
print("\nStrongest single features (held-out AUC; above 0.95 would be suspicious):")
print(single.head(5).round(3))

# 3) shuffled-label check
Xf = X.fillna(X.iloc[tr].median())
y_shuf = y.iloc[tr].sample(frac=1, random_state=0).values
rf = RandomForestClassifier(n_estimators=100, min_samples_leaf=20, n_jobs=-1, random_state=0).fit(Xf.iloc[tr], y_shuf)
print("\nShuffled-label AUC (expect about 0.5):", round(roc_auc_score(y.iloc[te], rf.predict_proba(Xf.iloc[te])[:, 1]), 3))

# 4) duplicate pages across train and test
if "content_id" in d.columns:
    print("Duplicate content_id across train/test:", len(set(d.content_id.iloc[tr]) & set(d.content_id.iloc[te])))
print("Clients shared across train/test:", len(set(d.client_id.iloc[tr]) & set(d.client_id.iloc[te])))


label-window columns: ['impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d']
rule-made / label-derived columns: ['age_tier', 'age_tier_order', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'health_score', 'needs_indexing', 'is_quick_win', 'needs_ctr_fix', 'needs_engagement_fix', 'ai_opportunity', 'is_underperformer', 'is_declining', 'is_initial_refresh_candidate']
identifier columns: ['client_id', 'content_id']
possible privacy columns (names only): none
excluded columns that leaked into X: none

Strongest single features (held-out AUC; above 0.95 would be suspicious):
clicks_prev_30d           0.968
impressions_prev_30d      0.961
sessions_prev_30d         0.930
content_age_days          0.676
days_since_last_update    0.641
dtype: float64

Shuffled-label AUC (expect about 0.5): 0.473
Duplicate content_id across train/test: 0
Clients shared across train/test: 0


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

**Excluded on purpose, and why:**
- Every `*_last_30d` column except the label source: they come from the window I am predicting.
- Rule-made columns (flags, health score, trend fields, tiers): built from the same data or the label, so they would leak it back to the model.
- `client_id` and `content_id`: used only to group the split and to check duplicates, never as features.
- Any column that could hold a name, address or search query: none is used, and none appears in outputs.

**Honest caveats:** `avg_position` covers 90 days and overlaps the label window. `content_age_days` and `days_since_last_update` are measured at the snapshot, so an edit during the label window could shift them. I could not rule this out from the table alone, so I treat results as directional.

**Reading the single-feature test:** `clicks_prev_30d` alone scores a held-out AUC of about 0.97, above my 0.95 flag. It comes from the *earlier* window, so I read this as persistence (pages with clicks last month tend to have clicks next month), not leakage. It is also the simple benchmark that any model must beat, and the later modeling notebook compares against it directly.


In [4]:
excluded = pd.DataFrame({
    "excluded group": ["label-window columns", "rule-made / label-derived columns", "identifier columns"],
    "columns found": [", ".join(label_window) or "none", ", ".join(rule_made) or "none", ", ".join(ids) or "none"],
    "reason": ["come from the window being predicted", "built from the same data or the label", "grouping and duplicate checks only"]})
excluded


,excluded group,columns found,reason
0,label-window columns,"impressions_last_30d, clicks_last_30d, session...",come from the window being predicted
1,rule-made / label-derived columns,"age_tier, age_tier_order, freshness_tier, word...",built from the same data or the label
2,identifier columns,"client_id, content_id",grouping and duplicate checks only


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.